In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import re

In [2]:
data = pd.read_csv('data/online_retail_II.csv')
data.shape

(1067371, 8)

### Business question: - Refine Later ###
* “How does price impact demand, and what price maximizes revenue?”
*  What is the optimal price point?
*  How sensitive are customers to price changes?
*  Do promotions actually increase revenue or just volume?

### Project Outline - Refine Later ###
1. EDA
    * Price distributions
    * Demand distibution (p vs. Q)
    * Promo comperitors
2. DGP
    * Endogeniety? 
    * Simultaneity? 
    * Promo treatment effects? 
3. Cleaning/Preprocessing
4. Feature Eng
    * Demand Signals
        * Rolling Ave
    * Price in context
    * Market in context
5. Elasticity Estimation
    * log-OLS
6. Addressing Endogeneity in Pricing
    * FE estimation
    * IV estimation
7. Validation
8. RevOpt
    * Simulate Rev/Profit curves
    * Identify Max and regionality within P/Q
9. A/B testing promos
    * OlS
    * DiD
10. Elasticity by segment
11. Demand & Revenue Curve Visualization
12. Streamlit? 


### Data Description ###
* InvoiceNo: Invoice number. Nominal. A 6-digit integral number uniquely assigned to each transaction. If this code starts with the letter 'c', it indicates a cancellation.
* StockCode: Product (item) code. Nominal. A 5-digit integral number uniquely assigned to each distinct product.
* Description: Product (item) name. Nominal.
* Quantity: The quantities of each product (item) per transaction. Numeric.
* InvoiceDate: Invoice date and time. Numeric. The day and time when a transaction was generated.
* Price: Unit price. Numeric. Product price per unit in sterling (Â£).
* CustomerID: Customer number. Nominal. A 5-digit integral number uniquely assigned to each customer.
* Country: Country name. Nominal. The name of the country where a customer resides.

In [3]:
data['Refund'] = np.where(data['Price'] < 0, 1, 0)
data['Return'] = np.where(data['Quantity'] < 0, 1, 0 )
returned_sales = data[(data['Return'] == 1) | (data['Refund'] == 1)]


In [4]:
returned_sales['Description'].value_counts(ascending=False).head(10)

Description
Manual                                537
REGENCY CAKESTAND 3 TIER              347
POSTAGE                               229
BAKING SET 9 PIECE RETROSPOT          211
STRAWBERRY CERAMIC TRINKET BOX        184
Discount                              172
WHITE HANGING HEART T-LIGHT HOLDER    135
check                                 123
WHITE CHERRY LIGHTS                   121
RED RETROSPOT CAKE STAND              111
Name: count, dtype: int64

In [5]:
sales_data = data[(data['Refund'] == 0) & (data['Return'] == 0)]
sales_data.shape

(1044416, 10)

In [6]:
sales_data['Clean_Desc'] = (
    sales_data['Description']
    .fillna("")
    .str.upper()
    .str.replace(r"[^A-Z\s]", " ", regex = True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

C:\Users\Kolby\AppData\Local\Temp\ipykernel_7104\123135938.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sales_data['Clean_Desc'] = (


In [7]:
top_desc = sales_data['Clean_Desc'].value_counts(ascending= False).head(25)
print("Top full descriptions:")
print(top_desc)


Top full descriptions:
Clean_Desc
WHITE HANGING HEART T LIGHT HOLDER    5783
REGENCY CAKESTAND TIER                4065
JUMBO BAG RED RETROSPOT               3395
ASSORTED COLOUR BIRD ORNAMENT         2939
PARTY BUNTING                         2742
LUNCH BAG BLACK SKULL                 2484
LUNCH BAG SUKI DESIGN                 2473
STRAWBERRY CERAMIC TRINKET BOX        2429
JUMBO STORAGE BAG SUKI                2400
FRENCH BLUE METAL DOOR SIGN           2303
HEART OF WICKER SMALL                 2299
JUMBO SHOPPER VINTAGE RED PAISLEY     2275
TEATIME FAIRY CAKE CASES              2257
PAPER CHAIN KIT S CHRISTMAS           2198
REX CASH CARRY JUMBO SHOPPER          2190
LUNCH BAG SPACEBOY DESIGN             2186
HOME BUILDING BLOCK WORD              2172
WOODEN FRAME ANTIQUE WHITE            2151
LUNCH BAG CARS BLUE                   2141
NATURAL SLATE HEART CHALKBOARD        2119
HEART OF WICKER LARGE                 2090
WOODEN PICTURE FRAME WHITE FINISH     2085
PACK OF PINK PAISLEY

In [8]:
def get_words (df, col):
    all_words = (
        df[col]
        .str.split()
        .explode()
    )

    stopwords = {
     "THE", "AND", "OF", "IN", "TO", "FOR", "ON", "WITH", "A", "AN",
     "SET", "PACK", "SMALL", "LARGE", "ASSORTED"
    }

    word_counts = (
        all_words[
            all_words.notna()
            & (all_words.str.len() > 2)
            & (~all_words.isin(stopwords))
        ]
        .value_counts()
        .head(50)
    )
    print("\nTop single words:")
    print(word_counts)
get_words(sales_data, 'Clean_Desc')


Top single words:
Clean_Desc
BAG           91439
RED           91005
HEART         78132
PINK          64048
RETROSPOT     57376
VINTAGE       54861
DESIGN        53286
WHITE         49966
BOX           49755
METAL         45041
CAKE          44892
CHRISTMAS     44073
BLUE          41588
HANGING       36187
LIGHT         35746
SIGN          35226
JUMBO         34759
HOLDER        34241
PAPER         30608
LUNCH         29986
GLASS         26645
TEA           26316
CARD          25569
DECORATION    24780
WOODEN        23468
CASES         23185
BOTTLE        22943
SPOTTY        22226
HOT           21839
WATER         21171
ROSE          20527
CERAMIC       20244
SPACEBOY      18267
MUG           18060
PAISLEY       17931
FAIRY         17247
BLACK         16985
TIN           16923
POLKADOT      16711
CREAM         16614
HOME          16495
GREEN         16177
SPOT          16144
PARTY         15366
GARDEN        15183
FELTCRAFT     15076
MINI          15036
LOVE          14496
BUNTING   

In [9]:
stopwords = {
     "THE", "AND", "OF", "IN", "TO", "FOR", "ON", "WITH", "A", "AN",
     "SET", "PACK", "SMALL", "LARGE", "ASSORTED"
    }

def get_bigrams(text: str) -> list[str]:
    words = [w for w in text.split() if len(w) > 2 and w not in stopwords]
    return [" ".join(pair) for pair in zip(words, words[1:])]

bigram_counter = Counter()

for desc in sales_data["Clean_Desc"]:
    bigram_counter.update(get_bigrams(desc))

top_bigrams = pd.Series(dict(bigram_counter)).sort_values(ascending=False).head(50)

print("\nTop bigrams:")
print(top_bigrams)


Top bigrams:
RED RETROSPOT        29615
METAL SIGN           27937
JUMBO BAG            26504
LIGHT HOLDER         23464
LUNCH BAG            21811
WATER BOTTLE         19696
HOT WATER            19696
CAKE CASES           18963
RED SPOTTY           14808
HANGING HEART        13033
FAIRY CAKE           11392
CHARLOTTE BAG        10570
DOLLY GIRL            9858
UNION JACK            9143
VINTAGE CHRISTMAS     8671
HEART LIGHT           8620
LUNCH BOX             8175
CAKE STAND            8051
BAG PINK              7976
DRAWER KNOB           7595
BAG RED               7520
BAG VINTAGE           7486
PINK POLKADOT         7448
PLASTERS TIN          7424
BIRTHDAY CARD         7258
RETRO SPOT            7187
HAND WARMER           7110
TRINKET BOX           6918
HEART DECORATION      6793
BAG SUKI              6672
SWEET HOME            6483
HOME SWEET            6472
CHAIN KIT             6442
PAPER CHAIN           6442
RIBBON REEL           6131
ALARM CLOCK           6101
ENGLISH ROSE  

### Need to clean categories - other under 20% ###

In [217]:
categories = {
    "Textiles": r"\b(TOWEL|NAPKIN|CUSHION COVER|PILLOW|RIBBON|THROW|FABRIC|LINEN|TEA TOWELS)\b",
    "Kitchen": r"\b(DISH|BOWL|SPOON|SPOONS|FORK|KNIFE|LUNCH BAG|CUP|MUG|CAKE STAND|WATER BOTTLE|TEACUP|CAKESTAND|CAKE CASES|PLATE|MILK JUG|BUTTER DISH|OVEN GLOVE|TEAPOT|KITCHEN SCALES|COOKIE CUTTERS|FRYING PAN|JAM MAKING)\b",
    "Storage": r"\b(BIN|BOX|STORAGE BAG|TIN|BASKET|BOTTLE|JAR|CRATE|HAMPER)\b",
    "Furniture": r"\b(TABLE|CHAIR|SOFA|BENCH|CUSHION|STOOL|DESK|DRAWER|SHELF)\b",
    "Toys": r"\b(DOLL|DOLLY|PUZZLE|GAME|BUILDING BLOCK|MARBLES|PLAYHOUSE|TOY)\b",
    "Accessories_bag": r"\b(JUMBO BAG|SHOPPER|PICNIC BAG|HAND WARMER|CHARLOTTE BAG|SHOULDER BAG|PEG BAG|TOTE BAG|AIRLINE BAG)\b",
    "Outdoors" : r"\b(PARASOL|UMBRELLA|WATERING CAN)\b",
    "Paper_goods": r"\b(NAPKIN|TISSUE|TISSUES|GIFT TAG|COAT HANGER|CARD|GIFT BAG|GIFT WRAP|PAPER CHAIN|PAPER DOILIES|NOTEBOOK|JOURNAL|ENVOLPE|WRAPPING|WRAP|STICKER|PAPER NAPKINS)\b",
    "Decor": r"\b(CLOCK|LIGHT|DOOR MAT|DOORMAT|METAL SIGN|PHOTO FRAME|PICTURE FRAME|DECORATION|BUNTING|DRAWER KNOB|HEART OF WICKER|ORNAMENT|HOOK HANGER|DOOR SIGN|CANDLEHOLDER|TRINKET POT|WALL ART|FIGURINE|STATUE)\b",
    "Crafts": r"\b(PENCIL|PENCILS|CHALK|CHALKBOARD|PEN|MARKER|SEWING|DRAWING SLATE)\b",   
}
sales_data["Category"] = "Other"

for category, pattern in categories.items():
    mask = sales_data["Clean_Desc"].str.contains(pattern, na=False, regex=True)
    sales_data.loc[mask, "Category"] = category

print("\nCategory counts:")
print(sales_data["Category"].value_counts())

C:\Users\Kolby\AppData\Local\Temp\ipykernel_7104\148812783.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sales_data["Category"] = "Other"
C:\Users\Kolby\AppData\Local\Temp\ipykernel_7104\148812783.py:16: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = sales_data["Clean_Desc"].str.contains(pattern, na=False, regex=True)
C:\Users\Kolby\AppData\Local\Temp\ipykernel_7104\148812783.py:16: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = sales_data["Clean_Desc"].str.contains(pattern, na=False, regex=True)
C:\Users\Kolby\AppData\Local\Temp\ipykernel_7104\14881278


Category counts:
Category
Other              408775
Decor              167732
Kitchen            124986
Storage             98762
Paper_goods         77206
Accessories_bag     63061
Crafts              33921
Toys                29287
Furniture           14951
Textiles            13662
Outdoors            12073
Name: count, dtype: int64


In [218]:
bigram_counter2 = Counter()

other_sales = sales_data[sales_data['Category'] == 'Other']

for desc in other_sales["Clean_Desc"]:
    bigram_counter2.update(get_bigrams(desc))

top_bigrams = pd.Series(dict(bigram_counter2)).sort_values(ascending=False).head(50)

print("\nTop bigrams:")
print(top_bigrams)


Top bigrams:
RED SPOTTY                5486
RED RETROSPOT             4389
FAIRY CAKE                4270
UNION JACK                4092
VINTAGE CHRISTMAS         3967
PANTRY DESIGN             3618
CAKE TINS                 3547
ROUND SNACK               3359
SNACK BOXES               3359
POPCORN HOLDER            3143
CHILDRENS APRON           3141
KEY FOB                   2766
MONEY BANK                2755
TINS PANTRY               2732
ANT WHITE                 2705
PARTY CANDLES             2681
ANTIQUE WHITE             2622
LOVE HEART                2622
SPACEBOY DESIGN           2385
CAKE MONEY                2371
ANTIQUE SILVER            2365
WHITE FINISH              2351
TEA GLASS                 2347
LUGGAGE TAG               2300
CHRISTMAS SCANDINAVIAN    2297
PIECE RETROSPOT           2255
CHRISTMAS CRAFT           2239
WHITE HEART               2223
FOOD COVER                2190
GLASS BOWLS               2168
LID GLASS                 2168
WOODEN FRAME             

In [219]:
get_words(other_sales, 'Clean_Desc')


Top single words:
Clean_Desc
HEART          30791
CHRISTMAS      26513
RED            24760
VINTAGE        21975
WHITE          20734
PINK           20613
RETROSPOT      20472
BLUE           15518
WOODEN         15389
DESIGN         14773
LIGHTS         11792
MINI           11116
GLASS          10909
WOOD           10741
CAKE           10256
TEA             9736
ROSE            9612
METAL           9368
GARDEN          9336
HANGING         8992
FLOWER          8958
CANDLE          8955
PAPER           8723
TINS            8482
FAIRY           8343
CANDLES         7991
CREAM           7977
RIBBONS         7924
BAG             7776
IVORY           7589
BOXES           7477
HOLDER          7273
PANTRY          7202
GARLAND         7141
ANTIQUE         7065
SPOTTY          7060
SILVER          6739
BUTTERFLY       6677
CUTLERY         6652
ROUND           6478
APRON           6399
CERAMIC         6386
BLACK           6359
STAR            6349
FELTCRAFT       6225
PIECE           6096
TRAD

In [220]:
lookup_sales = sales_data[(sales_data['Category'] == "Other")&(sales_data['Clean_Desc'].str.contains("HEART", na = False))]

bigram_counter3 = Counter()

for desc in lookup_sales["Clean_Desc"]:
    bigram_counter3.update(get_bigrams(desc))

top_bigrams = pd.Series(dict(bigram_counter3)).sort_values(ascending=False).head(50)

#print("\nTop bigrams:")
#print(top_bigrams)
print(lookup_sales['Clean_Desc'].value_counts(ascending = False).head(15))

Clean_Desc
RED WOOLLY HOTTIE WHITE HEART          1265
FELTCRAFT BUTTERFLY HEARTS             1243
HEART IVORY TRELLIS SMALL              1029
CREAM SWEETHEART MINI CHEST             966
HEART FILIGREE DOVE SMALL               935
HEART IVORY TRELLIS LARGE               890
WOODEN HEART CHRISTMAS SCANDINAVIAN     882
HANGING METAL HEART LANTERN             829
CERAMIC HEART FAIRY CAKE MONEY BANK     784
HEART FILIGREE DOVE LARGE               765
PLACE SETTING WHITE HEART               758
FOOD CONTAINER SET LOVE HEART           735
DOORSTOP RETROSPOT HEART                728
FOLKART ZINC HEART CHRISTMAS DEC        668
GINGHAM HEART DOORSTOP RED              648
Name: count, dtype: int64
